# 01.2 First Contact with the PayFlow Data Universe

> **Prerequisites:** 01.1 (rules vs learning; the late-payment problem and its modelling table)
> **What you'll learn:**
> - State the **grain** of a table and test it, instead of trusting the name of its id column
> - Predict what a join does to row count *before* running it, and detect fan-out and silent loss
> - Recognize a column whose unit varies by row, and why aggregating it produces no error
> - Write an ingestion contract that refuses bad data at the door rather than downstream
> - Reconcile a derived business number against cash, and read the residual
> **Level:** Beginner · **Series:** 01 The ML Landscape & Project Lifecycle

> ⚡ **Thursday 2026-02-05, 08:15** — the board deck goes out claiming record January
> billings. By 09:30 the finance lead cannot reconcile it with the bank: the deck is high by a
> factor of thirty-four. No job failed, no alert fired, no cell went red. The cause: a `SUM()`
> over a column whose unit changes from one row to the next.

## Concept
### Plain-English Explanation

In 01.1 a modelling table appeared as if by magic: `lab.build_dataset()` returned invoices
joined to payments and customers, with a `late` label attached, and the notebook got on with
comparing a rule against a model. That function was doing a great deal of quiet work, and every
line of it was there because something in the raw export would otherwise have produced a
confident wrong answer.

This notebook opens the box. PayFlow's data universe is six CSV exports — customers,
subscriptions, invoices, payments, support tickets, add-on purchases — and they are messy in the
specific ways that operational systems are always messy: an invoicing bug that reposted some
rows, a legacy payment gateway that writes dates and numbers in its own format, a currency
column that most reports forget exists. None of this is exotic. All of it is what "the data" is,
before anybody has been kind to it.

The skill being built is not cleaning — that is series 09's subject. It is **reading a table
correctly in the first place**: knowing what one row means, what happens when you join it to
another, and which columns are lying about their units.

### Technical Explanation

The central concept is **grain**: the statement of what exactly one row represents. "One row per
invoice" is a grain. So is "one row per payment", which is *not* the same as one row per invoice
even though both tables have an `invoice_id`. Grain is a claim, and like any claim it can be
false — PayFlow's invoice export declares one row per invoice and ships 288,936 rows carrying
only 288,040 distinct `invoice_id` values, because a reposting bug duplicated 896 of them
(`_data/SPEC.md` rule M1).

Grain determines join behaviour, which is where the damage happens. Joining a table at invoice
grain to one at payment grain produces a result at payment grain, and any per-invoice statistic
computed afterwards silently double-counts every invoice that was paid in instalments. The
industry name for this is **fan-out**. Its mirror image is **silent loss**: an inner join
discards rows whose key has no match, so joining tickets to customers quietly deletes every
ticket that arrived without a `customer_id` — which, in PayFlow, is almost exactly the set of
spam tickets (M14).

The third failure mode is **unit heterogeneity**. The `amount` column on an invoice is a number
whose meaning depends on the `currency` column beside it: 5,000 means one thing for a US
customer and something ninety times smaller for an Indian one. Summing that column across
currencies is not an approximation, it is a category error, and no type system, null check or
schema validator in the default pipeline will object. ⭐ **CRITICAL CONCEPT** — a money value is
the *pair* (amount, currency). A schema that stores only the number has already lost the
information needed to add two rows together, and every downstream consumer is left to remember
a rule that nothing enforces.

The numbers that matter for orientation: 8,060 customers, 288,936 invoices spanning 2019 to
2026, 286,432 payments, 43,706 support tickets, 18,075 add-on orders, across seven currencies.

### Mental Model

A table is a claim about what one row means. Every bug below — the duplicated invoices, the
fan-out, the vanished spam, the thirty-four-fold revenue error — is that claim being false while
every column still looks perfectly reasonable.

## How It Works

Six exports, four of which are related through two keys. The arrows below are labelled with
cardinality, because cardinality is what predicts a join's effect on row count:

```text
  customers (8,060)                     one row per customer          key: customer_id
      |  1
      |------------------< subscriptions (9,261)     1 customer -> many periods
      |  1
      |------------------< invoices (288,936)        1 customer -> many invoices
      |                        |  1
      |                        |----< payments (286,432)   1 invoice -> 0..n payments  <- fan-out
      |  1                                                  (0 when never paid, 2 when split)
      |------------------< support_tickets (43,706)  customer_id NULLABLE  <- silent loss on
      |  1                                            inner join (spam has no customer)
      |------------------< addon_purchases (18,075)

  invoices.amount  ---- means nothing without ---->  invoices.currency
       (a number)                                     (INR | USD | AED | EUR | SGD | GBP | AUD)
```

Two mechanisms follow from that picture.

**Join arithmetic.** An inner join between invoices and payments changes the row count in two
directions at once. Invoices with no payment disappear; invoices with two payments become two
rows. In PayFlow those effects are 12,529 rows removed and 11,822 rows added, which net out to
a change of −1,603 rows, or −0.55%. That near-zero net is the trap: a pipeline that asserts "row
count changed by less than one percent after the join" passes, while roughly twenty-four
thousand rows moved underneath it. Row count is not a join check. Key cardinality is.

**Unit arithmetic.** Summing `amount` is a weighted sum in which the weights are exchange rates
nobody applied. Because the Indian customer base is both the largest by count and denominated in
the numerically largest currency, it dominates: INR contributes 97.7% of the raw total but only
38.0% of the true one. The reported figure is therefore not "slightly off" or "noisy" — it is a
number with no unit at all, and it happens to land 34.21 times too high.

## Hands-On Build
### Stage A — from scratch

No pandas. A CSV is a text file, and every dtype you later rely on is an interpretation someone
chose to apply. Reading with the standard library makes that concrete, and it lets us establish
the grain of the invoice table by counting keys ourselves — which is exactly what a `nunique()`
call does later, minus the mystery.

In [1]:
# Load the committed lab module; it owns the file handling and the experiment functions.
import csv
import importlib.util
import sys
from collections import Counter
from pathlib import Path

LAB = Path.cwd() / "_lab" / "lab_01.2_data_universe.py"
spec = importlib.util.spec_from_file_location("lab_01_2", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_01_2"] = lab
spec.loader.exec_module(lab)

lab.peek_raw("invoices.csv.gz")

# Establish the grain by hand: does one row really mean one invoice?
with lab.open_text(lab.RAW / "invoices.csv.gz") as fh:
    rows = list(csv.DictReader(fh))
ids = Counter(r["invoice_id"] for r in rows)
repeats = {i: c for i, c in ids.items() if c > 1}
print(f"rows={len(rows):,}   distinct invoice_id={len(ids):,}   "
      f"ids appearing more than once={len(repeats):,}")

# And check the claim that `amount` is a number.
commas = [r["amount"] for r in rows if "," in r["amount"]]
print(f"amounts carrying a thousands separator: {len(commas):,}   e.g. {commas[:3]}")
print(f"float({commas[0]!r}) would raise ValueError - the column is text that looks numeric")

  invoices.csv.gz
    columns (8): invoice_id, customer_id, issue_date, due_date, currency, amount, status, reminder_count
    ['INV-0000001', 'CUST-10000', '2019-05-08', '2019-05-23'] ...
    ['INV-0000002', 'CUST-10000', '2019-06-08', '2019-06-23'] ...
    ['INV-0000003', 'CUST-10000', '2019-07-08', '2019-07-23'] ...
    every value above is a str - dtype is an interpretation pandas applies,
    not a property the file has



rows=288,936   distinct invoice_id=288,040   ids appearing more than once=896
amounts carrying a thousands separator: 3,792   e.g. ['4,554.29', '4,753.31', '1,626.94']
float('4,554.29') would raise ValueError - the column is text that looks numeric


The declared grain is false: 288,936 rows carry 288,040 distinct ids, so 896 identifiers appear
more than once. Nothing about the file announces this. The invoice export looks exactly like a
well-formed table, and a `GROUP BY customer_id` over it will overstate 896 invoices' worth of
revenue for the customers unlucky enough to be affected.

The `amount` column tells the second story. Three thousand seven hundred and eighty-nine values
carry a thousands separator, a legacy of exports written before 2020-07, and `float("4,554.29")`
raises. Note carefully what this means: the column is *text that looks numeric*. Everything
downstream depends on which of those two descriptions the reader believes.

### Stage B — idiomatic

Now pandas, doing the same work with less ceremony — and immediately doing more, because the
grain question applies to all six tables at once and the join question cannot be asked without
it. The parity check is direct: `nunique()` must reproduce the hand count above.

In [2]:
import pandas as pd

# Same question as Stage A, asked of every table in the universe at once.
frames = {name: lab.grain_report(name, key, declared)
          for name, (declared, key) in lab.TABLES.items()}

  customers.csv              n=   8,060  unique customer_id=   8,060  exact dup rows=    0  OK   (one row per customer)
  subscriptions.csv          n=   9,261  unique sub_id=   9,261  exact dup rows=    0  OK   (one row per subscription period)


  invoices.csv.gz            n= 288,936  unique invoice_id= 288,040  exact dup rows=  896  GRAIN VIOLATED   (one row per invoice)


  payments.csv.gz            n= 286,432  unique payment_id= 286,432  exact dup rows=    0  OK   (one row per payment)
  support_tickets.csv.gz     n=  43,706  unique ticket_id=  43,706  exact dup rows=    0  OK   (one row per ticket)


  addon_purchases.csv.gz     n=  18,075  unique order_id=  18,075  exact dup rows=    0  OK   (one row per add-on order)


`invoices.csv.gz` reports 288,936 rows against 288,040 unique ids — identical to the Stage A
hand count, which is the parity check. Every other table holds its grain. That distinction
matters more than it looks: five of six tables can be trusted to mean what they say, and knowing
*which* one cannot is the difference between a correct aggregate and a plausible one.

With the grain of each side established, the join becomes predictable rather than surprising.

In [3]:
lab.join_experiments(frames["invoices.csv.gz"], frames["payments.csv.gz"],
                     frames["support_tickets.csv.gz"], frames["customers.csv"])

  invoices(288,936) x payments(286,432) inner join -> 287,333 rows   (net change -1,603, -0.55%)
    but that small net hides two large opposite effects:
      - 12,529 invoices DROPPED (never paid: overdue/disputed/written off)
      - 11,822 extra rows ADDED by fan-out (10,921 invoices have >1 payment, SPEC M13)
    fan-out factor 1.043 -> any per-invoice average computed on this frame double-counts the split-paid invoices


    aggregate-then-join -> 288,936 rows, grain preserved (True)
  tickets(43,706) x customers inner join -> 41,642 rows, 2,064 silently dropped
    of the dropped, 2,064 are spam tickets with no customer_id (SPEC M14) -> a spam classifier trained on the join never sees its own positive class


⚠️ The invoice-to-payment join shrinks the frame by 1,603 rows, a change of −0.55%, and that
number is a lie of composition: 12,529 invoices were dropped because they were never paid, and
11,822 rows were added because 10,921 invoices were settled in two instalments. A guard that
watches total row count sees a rounding error. The `validate="m:1"` argument on the
aggregate-then-join version is the real guard — it makes pandas raise when the right-hand key is
not unique, turning a silent fan-out into an exception at the line that causes it.

The ticket join is the more alarming one, because it loses 2,064 rows and *every one of them is
a spam ticket*. Spam arrives from outside PayFlow and therefore has no `customer_id`, so an
inner join to the customer table deletes precisely the rows a spam classifier exists to detect.
A model trained on that joined frame would be trained on data from which its entire positive
class had been removed, would score superbly on a test set drawn the same way, and would fail on
first contact with reality. Series 29 builds that classifier; it will read tickets before
joining them, for this reason.

### Stage C — production

The lesson of both failures is that correctness here cannot rest on remembering. The ingestion
job should refuse malformed data at the door, and the money representation should turn the
cold open's incident into a `TypeError` rather than a number.

In [4]:
from dataclasses import dataclass
from decimal import Decimal


class CurrencyMismatch(TypeError):
    """Raised when two Money values of different currencies are combined."""


@dataclass(frozen=True)
class Money:
    """A money value is the PAIR (amount, currency). Storing only the number is the bug.

    Decimal rather than float because binary floating point cannot represent 0.10 exactly,
    and money is compared for equality by auditors.
    """
    amount: Decimal
    currency: str

    def __add__(self, other: "Money") -> "Money":
        if self.currency != other.currency:
            raise CurrencyMismatch(
                f"refusing to add {self.currency} to {other.currency}; convert first")
        return Money(self.amount + other.amount, self.currency)

    def to_usd(self, rate_per_usd: Decimal) -> "Money":
        return Money(self.amount / rate_per_usd, "USD")


ledger = [Money(Decimal("4554.29"), "INR"), Money(Decimal("612.00"), "USD")]
try:
    total = ledger[0] + ledger[1]
except CurrencyMismatch as exc:
    print(f"blocked at the type level: {exc}")

rates = {"INR": Decimal("88.0"), "USD": Decimal("1")}
converted = [m.to_usd(rates[m.currency]) for m in ledger]
print(f"after explicit conversion: ${sum(m.amount for m in converted):,.2f} USD")

blocked at the type level: refusing to add INR to USD; convert first
after explicit conversion: $663.75 USD


The board-deck error cannot occur against this type. Adding INR to USD is not a judgement call
that a careful analyst gets right and a rushed one gets wrong; it raises. That is the difference
between a convention and a constraint, and only one of them survives staff turnover.

The second half of the production answer is a contract the ingestion job runs before anything
downstream is allowed to read the export — evaluated next.

## Evaluation

For a data notebook the thing being measured is the data itself, so the harness is a set of
contract checks over the raw export and the metric is how many pass, plus a reconciliation of
the derived business number against cash actually collected. The baseline to beat is the naive
pipeline that produced the incident: it applies zero checks and lands 34.21 times high. A
meaningful result is binary for the contract — any FAIL blocks ingestion — and for the
reconciliation is a residual small enough to be explained by known FX movement rather than by a
bug, which for PayFlow means roughly one percent or less.

In [5]:
lab.check_contract(frames["invoices.csv.gz"], frames["payments.csv.gz"],
                   frames["customers.csv"])
print()
lab.reconcile(frames["invoices.csv.gz"], frames["payments.csv.gz"])

  [FAIL] invoice_id is unique                   offending rows:      896
  [FAIL] no exact duplicate rows                offending rows:      896
  [PASS] amount parses as a number              offending rows:        0
  [FAIL] amount is already numeric dtype        offending rows:  288,936
  [PASS] currency present on every row          offending rows:        0
  [FAIL] status uses one casing                 offending rows:   28,239
  [PASS] due_date >= issue_date                 offending rows:        0
  [PASS] payment.invoice_id exists in invoices  offending rows:        0
  [PASS] customer_id exists in customers        offending rows:        0

  contract score: 5/9 checks pass on the raw export



  collected (from payments, each at its own settlement rate): $   438,685,003
  billed on those same invoices (median-rate table):          $   436,270,391
  gap: $2,414,612 (+0.55%) - FX moved between issue and settlement; a reconciliation this tight is the signal the conversion is right


Five of nine checks pass. The four failures are precisely the defects this notebook has been
tracing: 896 duplicate rows breaking the declared grain twice over, all 288,936 rows failing the
"already numeric" check because of the comma formatting, and 28,239 rows carrying `status` in
inconsistent casing so that `status == "paid"` quietly misses a tenth of the paid invoices. None
of these would raise. All of them would corrupt a downstream aggregate.

The reconciliation is the part worth internalizing. Collected cash converted at each payment's
own settlement rate totals $438,685,003; the same invoices billed and converted through a
median-rate table total $436,270,391. The residual is $2,414,612, or +0.55%, and it has a known
cause — exchange rates move between the day an invoice is issued and the day it settles, and the
median-rate table cannot capture that. A residual of this size, with a mechanism attached, is
evidence the conversion is right. A residual of 34.21 times, with no mechanism, is the incident.
Deriving a number two independent ways and explaining the gap is the cheapest correctness check
available in data work, and it is the one most often skipped.

## Design Patterns / Tradeoffs

**Money as a typed pair versus a bare numeric column.** A bare `amount` column with a companion
`currency` column is what PayFlow ships, and it is the cheapest thing to write: every insert is
a number, every existing report keeps working, and storage is minimal. Its failure mode is
exactly this notebook — correctness depends on every consumer, forever, remembering to join on
the currency, and there is no point at which the system can tell them they forgot. A typed
`Money` value, or the common database equivalent of storing `amount_minor`, `currency` and a
frozen `amount_usd` computed at write time, makes the wrong operation impossible instead of
merely discouraged. The cost is real: three columns instead of one, a rate source at write time,
and a migration for everything already written. Use the typed representation whenever more than
one currency exists or ever might; keep the bare column only in a genuinely single-currency
system, and treat "we are single-currency" as an assumption with an expiry date.

**Fail-fast contract versus quarantine-and-continue.** The contract above blocks ingestion on any
failure, which is the right default for financial data: a report that does not run is a visible
problem, while a report that runs on corrupt input is an invisible one. The alternative routes
offending rows to a quarantine table and processes the remainder, which keeps the pipeline
moving and is appropriate for high-volume telemetry where partial data is still useful and
lateness is worse than incompleteness. The trap in the quarantine pattern is that quarantine
tables are write-only in practice unless someone owns the queue and its size is alerted on — an
unwatched quarantine is just deletion with extra steps. For PayFlow's invoices, fail fast; for
support-ticket ingestion, quarantine with an alert on queue depth is the better trade.

**Recommendation:** fail-fast contract at the ingestion boundary, typed money end to end, and
the reconciliation above running nightly as a scheduled test rather than as an investigation
performed after a board meeting. The condition that would change it is volume — if invoice
ingestion grew to a rate where blocking on a single bad row stalled a real-time ledger, the
contract moves to quarantine-with-alert and the reconciliation becomes the primary guard.

## Production Scenario
### Symptoms

**Thursday 2026-02-05, 08:15.** The January board deck is circulated. The billings headline is
an all-time record. What the on-call analyst sees, in order:

- **08:15** — the deck reports January billings that no one recognizes; the number is roughly
  thirty-four times the figure the finance lead reconciles from bank settlements.
- The revenue-by-country panel shows India at 97.7% of billings. India is the largest market by
  customer count, but the account executives know it is not three-quarters of revenue, let alone
  nineteen-twentieths.
- The pipeline is **green**. The nightly job ran, wrote its output, and logged no warning. Row
  counts are normal. No null-rate check moved.
- Month-over-month "growth" is large in a month when customer count was flat, and the growth
  tracks the share of new Indian customers rather than anything about price or volume.
- An earlier reviewer of the same job had once seen the total come back as an enormous
  unparseable string and had "fixed" it by casting the column.

In [6]:
lab.currency_incident(frames["invoices.csv.gz"], frames["payments.csv.gz"])

  invoices.amount dtype as read: <StringDtype(na_value=nan)>  (SPEC M7: legacy exports carry '4,554.29')
    .sum() returned str: 496.31494.83546.47540.66560.45542.54545.... (len 1,366,049)
    -> the column concatenated. No exception, no warning, no number.



  billings, summed the way the 2019 report does it:   15,897,163,634
  billings, converted to USD first:                       464,644,960
  overstatement factor: 34.21x (+$15,432,518,674 of currency that does not exist)

  where the phantom revenue comes from:
            rows            raw         usd share_of_raw share_of_usd
currency                                                             
INR      116,888 15,525,697,689 176,433,395        97.7%        38.0%
USD       62,456    114,809,776 114,809,776         0.7%        24.7%
AED       19,129    106,537,627  29,023,790         0.7%         6.2%
EUR       31,814     51,464,039  56,560,105         0.3%        12.2%
SGD       18,039     42,354,020  32,091,241         0.3%         6.9%
GBP       28,683     29,763,845  38,153,884         0.2%         8.2%
AUD       11,927     26,536,638  17,572,769         0.2%         3.8%


### Diagnosis

Walking the observability ladder, in the data-pipeline form it takes for a reporting job:

1. **Alert** — reconciliation break between the reported billings figure and bank settlement.
   Business symptom only; candidate causes are a double-counted join, a duplicated source file,
   a currency error, or a genuine record month.
2. **Output dashboards** — the by-country decomposition shows INR at 97.7% of the raw total
   against 38.0% of the converted total. The error is concentrated in one currency rather than
   spread across all rows, which eliminates "the job ran twice" and "the source file was
   duplicated": either of those would inflate every country proportionally.
3. **Job logs** — clean. No exception, no retry, no warning. This eliminates the whole class of
   crash-and-partial-write hypotheses and points at a computation that is wrong while being
   perfectly well-formed.
4. **Input-data checks** — schema unchanged, no nulls in `amount` or `currency`, row count in
   the normal range, freshness fine. The inputs are not the problem; the interpretation is.
5. **Column-level inspection** — `amount` reads as `StringDtype` because of the comma-formatted
   legacy rows, and `.sum()` on it returns a 1,366,049-character concatenation rather than a
   number. This is the second, independent bug in the same column, and it explains the earlier
   reviewer's "enormous string": their cast fixed the symptom and left the unit error untouched.
6. **Version diff** — the reporting query has not changed. What changed, years ago, is the
   business: the query was written when PayFlow billed only in USD, and the multi-currency
   rollout added a `currency` column without revisiting a single consumer of `amount`.

### Root Cause

`invoices.amount` is denominated in each customer's local currency, and the billings query sums
it without conversion, so every row contributes its face value regardless of unit. Because INR
is both the highest-count currency and numerically the largest per unit, it supplies 97.7% of
the raw sum against 38.0% of real revenue, and the reported total lands 34.21 times too high —
$15,897,163,634 against an actual $464,644,960.

### Fix

**Mitigation now.** Restate the deck from the converted total, and take the billings panel out
of the dashboard until the query is corrected — a missing number is recoverable, a wrong number
that people plan against is not. Publish the per-currency decomposition alongside the restated
figure so the audience can see the correction rather than being asked to trust a second number
from the same source that produced the first.

**Permanent fix.** Two changes, and the second is the one that matters. First, the query converts
before it aggregates, using a rate table with an effective date rather than a single current
rate. Second — because a corrected query is still a convention that the next engineer can
forget — the invoice schema gains a frozen `amount_usd` written at issue time next to
`(amount, currency)`, and application code moves to the `Money` type from Stage C so that adding
mismatched currencies raises instead of returning a number. The contract check
`amount is already numeric dtype` also becomes blocking, which removes the string-concatenation
failure at the same boundary.

### Prevention

- **Nightly reconciliation as a test, not an investigation.** Bill-versus-collect in USD, failing
  the build when the residual exceeds one percent. On this data the healthy residual is +0.55%
  with a known FX explanation, so the threshold has real headroom and a real meaning.
- **Ban unqualified aggregation of `amount`.** A lint or review rule that rejects any `SUM(amount)`
  not accompanied by a `GROUP BY currency` or an explicit conversion. Cheap, mechanical, and it
  catches the class rather than the instance.
- **Make the contract blocking at ingestion**, including the numeric-dtype check that would have
  caught the string column on the day the first comma-formatted export arrived.
- **Treat "we are single-currency" as a dated assumption.** The bug was introduced not by the
  multi-currency rollout but by the absence of any inventory of what depended on the old
  assumption. Any schema change that adds a qualifier column should come with a list of the
  consumers of the column it qualifies.

## Common Pitfalls

⚠️ **Aggregating a column whose unit varies by row.** No error, no null, no warning — just a
number with no meaning. The tell is a column that only makes sense next to another column;
whenever you find one, the aggregate needs a conversion or a `GROUP BY` of the qualifier.

⚠️ **Arithmetic on a string column that looks numeric.** Under pandas 3 the comma-formatted
`amount` reads as `StringDtype`, and `.sum()` concatenates 288,936 values into a 1,366,049
character string instead of raising. Assert dtypes after reading; never infer them from the
column name.

**Using row count as a join check.** The invoice-to-payment join moved 24,351 rows for a net of
−1,603. Check key cardinality with `validate=` instead, which raises at the join rather than
leaving a wrong average to be discovered later.

**Inner-joining on a nullable key.** Every one of the 2,064 tickets deleted by the customer join
is a spam ticket. When a foreign key is nullable, the null population is almost always a
meaningful subgroup rather than an accident, so ask what it consists of before choosing the join
type.

**Trusting an id column because it is named like a key.** `invoice_id` repeats on 896 rows.
Grain is a property to be tested on arrival, not inferred from naming.

**Case-sensitive comparison against a free-text status.** `status == "paid"` misses 28,239 rows
carrying other casings, and the resulting undercount looks entirely plausible — which is what
makes it dangerous.

**Cleaning ad hoc inside each analysis.** The same fixes rewritten per notebook drift apart, and
two dashboards disagree with no way to say which is right. Fixes belong in one ingestion layer
with a contract; series 09 builds it properly.

## Interview Questions

1. **Derive this.** Given a left table at grain A with `n_a` rows and a right table whose key
   has an average multiplicity `m` over matched keys, derive the row count of an inner join, and
   show why net row-count change cannot detect fan-out. *Answer shape:* result rows equal the sum
   of multiplicities over matched keys; unmatched left rows subtract while multiplicities above
   one add, so the two effects offset and the net can be near zero — PayFlow's −1,603 net hides
   12,529 removed and 11,822 added.
2. **Design this.** Design the ingestion layer for these six exports so that a downstream analyst
   cannot produce the board-deck error. *Answer shape:* contract at the boundary with blocking
   checks on grain, dtype and referential integrity; typed money or a frozen `amount_usd` written
   at ingest; one cleaned layer that all consumers read; nightly reconciliation as a failing test;
   quarantine with alerting where lateness beats incompleteness.
3. **Debug this.** A monthly revenue figure is 34 times too high, the pipeline is green, row
   counts are normal and nothing is null. Walk your diagnosis. *Answer shape:* decompose the
   aggregate by every categorical dimension to localize the error — concentration in one currency
   rules out double-processing, which would inflate uniformly; then inspect the column's dtype and
   unit; then diff the query against the schema history to find the assumption that expired.
4. Why is a spam classifier trained on `tickets ⋈ customers` guaranteed to disappoint, and what
   would its offline metrics look like? *Answer shape:* the join removes rows with null
   `customer_id`, which is most of the spam; offline scores look excellent because the test set
   inherits the same filter, and the failure appears only in production where unjoined tickets
   arrive.
5. What is the difference between a convention and a constraint here, and which failures does each
   prevent? *Answer shape:* a convention is "remember to convert before summing" and fails at the
   first new engineer or rushed deadline; a constraint is a type or a blocking check that makes
   the wrong operation impossible — prefer constraints at boundaries where the cost of being wrong
   is high and the error is silent.
6. You compute a business number two independent ways and they differ by 0.55%. What do you do?
   *Answer shape:* do not accept or reject on size alone — find the mechanism. Here FX moves
   between issue and settlement, which predicts a small signed residual; a gap with a mechanism is
   evidence of correctness, a gap without one is an open bug regardless of magnitude.
7. The invoice export has 896 duplicate rows. Give two plausible upstream causes and the
   different fix each implies. *Answer shape:* an at-least-once delivery or retry producing exact
   duplicates, fixed by idempotent writes keyed on invoice id; versus a genuine business event
   such as a reissue, in which case the rows are not duplicates at all and the grain statement
   itself is wrong and needs a version column.

## Key Takeaways

- State the grain of every table on arrival and test it; PayFlow's invoice export declares one
  row per invoice and violates it on 896 rows.
- Predict a join's cardinality before running it, and guard with `validate=` — row count is not a
  join check, since 24,351 moved rows netted to −1,603 here.
- An inner join on a nullable foreign key deletes a subgroup, and the subgroup is usually
  meaningful: all 2,064 lost tickets were spam.
- A column whose meaning depends on a neighbouring column cannot be aggregated alone; summing
  `amount` across currencies landed 34.21 times high with no error raised.
- Assert dtypes after reading, because text that looks numeric concatenates instead of summing
  and produces no warning at all.
- Prefer constraints to conventions at boundaries: a `Money` type turns the board-deck incident
  into an exception at the offending line.
- Derive important numbers two ways and explain the residual; +0.55% with a known FX mechanism is
  evidence, while a large gap with no mechanism is an open bug.
- Fix data at one ingestion layer with a contract, never per notebook, or dashboards will
  disagree with no authority to settle them.

## Related

**Backward**

- **01.1 Rules or Learning?** — its `build_dataset` silently performed the duplicate-drop,
  comma-parse, first-payment aggregation and currency conversion justified here; the leakage
  exclusions it made are revisited in 12.

**Forward**

- **01.3 Reproducibility as an Engineering Contract** — extends the ingestion contract idea to
  the whole run: pinned versions, seeds and a manifest including the data hashes from
  `_data/SPEC.md`.
- **01.5 Problem Framing** — the label built on top of this universe, and why the choice of
  horizon and population is a design decision rather than a detail.
- **03.1 Pandas & Modern DataFrames** — the `str` dtype, copy-on-write and the join semantics
  used here are made canonical, including the 2019 to 2026 migration this data keeps triggering.
- **09.1 Data Cleaning & Pre-processing** — owns cleaning as a subject: imputation for the
  sentinel values, deduplication strategy for the near-duplicate customers, and the leakage-safe
  ordering of every step.
- **12.1 Model Evaluation & Validation** — why a test set drawn through the same broken join as
  the training set cannot detect the problem.
- **23.1 Anomaly Detection & Association Rules** — consumes `addon_purchases` at its natural
  grain, where basket construction depends on exactly the cardinality reasoning built here.
- **29.1 Natural Language Processing** — builds the ticket classifier, reading tickets before
  joining them so the spam class survives.
- **34.1 ML in Production** — turns the nightly reconciliation and blocking contract into
  monitored jobs with ownership and alert thresholds.